# ポートフォリオゲーム ボラティリティ分析

**J-Quants API** を使用して指定30銘柄の株価・ボラティリティ・相関を分析する。

## 前提
1. リポジトリ直下に `.env` を作成して以下を記入：
   ```
   JQUANTS_EMAIL=your_email@example.com
   JQUANTS_PASSWORD=your_password
   ```
2. 必要パッケージ
   ```bash
   pip install pandas numpy matplotlib seaborn requests python-dotenv
   ```

参考: https://jpx-jquants.com/

## 1. セットアップ

In [ ]:
import os
import time
import json
from datetime import datetime, timedelta
from pathlib import Path

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

# matplotlibで日本語フォントを使えるように（環境によっては要調整）
plt.rcParams['font.family'] = ['Hiragino Sans', 'Yu Gothic', 'Meiryo', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

# .env 読込（リポジトリ直下を想定）
ROOT = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
load_dotenv(ROOT / '.env')

EMAIL = os.getenv('JQUANTS_EMAIL')
PASSWORD = os.getenv('JQUANTS_PASSWORD')
assert EMAIL and PASSWORD, '.env に JQUANTS_EMAIL と JQUANTS_PASSWORD を設定してください'
print('credentials loaded')

## 2. J-Quants 認証

1. メール／パスワード → **リフレッシュトークン** を取得
2. リフレッシュトークン → **IDトークン** を取得（APIリクエストに使用）

In [ ]:
BASE_URL = 'https://api.jquants.com/v1'

def get_refresh_token(email: str, password: str) -> str:
    resp = requests.post(
        f'{BASE_URL}/token/auth_user',
        data=json.dumps({'mailaddress': email, 'password': password}),
    )
    resp.raise_for_status()
    return resp.json()['refreshToken']

def get_id_token(refresh_token: str) -> str:
    resp = requests.post(
        f'{BASE_URL}/token/auth_refresh?refreshtoken={refresh_token}',
    )
    resp.raise_for_status()
    return resp.json()['idToken']

refresh_token = get_refresh_token(EMAIL, PASSWORD)
id_token = get_id_token(refresh_token)
HEADERS = {'Authorization': f'Bearer {id_token}'}
print('authenticated')

## 3. 指定30銘柄を読み込み

In [ ]:
stocks = pd.read_csv(ROOT / 'data' / 'stocks.csv')
stocks['code_str'] = stocks['code'].astype(str).str.zfill(4)
# J-Quants は4桁 or 5桁コード。4桁を5桁化（末尾0付与）した形式が使われる
stocks['jq_code'] = stocks['code_str'] + '0'
print(f'{len(stocks)} 銘柄ロード')
stocks.head()

## 4. 日次株価データ取得

`/prices/daily_quotes` エンドポイント。
- 過去 **1年分** を取得（必要に応じて変更）
- 無料プランは12週間の遅延があるため、最新データはプランに依存

In [ ]:
def fetch_daily_quotes(code: str, from_date: str, to_date: str) -> pd.DataFrame:
    """単一銘柄の日次株価を取得（ページング対応）"""
    rows = []
    pagination_key = None
    while True:
        params = {'code': code, 'from': from_date, 'to': to_date}
        if pagination_key:
            params['pagination_key'] = pagination_key
        resp = requests.get(f'{BASE_URL}/prices/daily_quotes', headers=HEADERS, params=params)
        resp.raise_for_status()
        data = resp.json()
        rows.extend(data.get('daily_quotes', []))
        pagination_key = data.get('pagination_key')
        if not pagination_key:
            break
    return pd.DataFrame(rows)

to_date = datetime.today().strftime('%Y-%m-%d')
from_date = (datetime.today() - timedelta(days=365)).strftime('%Y-%m-%d')
print(f'期間: {from_date} 〜 {to_date}')

all_data = []
for _, row in stocks.iterrows():
    code = row['jq_code']
    try:
        df = fetch_daily_quotes(code, from_date, to_date)
        if not df.empty:
            df['name'] = row['name']
            df['sector'] = row['sector']
            all_data.append(df)
        print(f"  {code} {row['name']}: {len(df)}件")
    except Exception as e:
        print(f"  {code} {row['name']}: ERROR {e}")
    time.sleep(0.2)  # レート制限対策

prices = pd.concat(all_data, ignore_index=True)
prices['Date'] = pd.to_datetime(prices['Date'])
print(f'\n合計 {len(prices)} 行')

## 5. 日次リターン算出

In [ ]:
# 終値ベース（調整後終値 AdjustmentClose があればそれを優先）
close_col = 'AdjustmentClose' if 'AdjustmentClose' in prices.columns else 'Close'

wide = prices.pivot_table(index='Date', columns='name', values=close_col).sort_index()
returns = wide.pct_change().dropna(how='all')
print(f'価格パネル shape={wide.shape} / リターン shape={returns.shape}')
returns.tail()

## 6. ヒストリカル・ボラティリティ（年率）

$\sigma_{annual} = \sigma_{daily} \times \sqrt{252}$

In [ ]:
TRADING_DAYS = 252
annual_vol = returns.std() * np.sqrt(TRADING_DAYS)
annual_ret = returns.mean() * TRADING_DAYS
sharpe = annual_ret / annual_vol  # 無リスク金利 0% 想定

name_to_sector = dict(zip(stocks['name'], stocks['sector']))
summary = pd.DataFrame({
    'sector': annual_vol.index.map(name_to_sector),
    'annual_return': annual_ret,
    'annual_volatility': annual_vol,
    'sharpe': sharpe,
}).sort_values('annual_volatility', ascending=False)
summary.style.format({'annual_return': '{:.2%}', 'annual_volatility': '{:.2%}', 'sharpe': '{:.2f}'})

## 7. 可視化

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
colors = {'エンタメ': '#22c55e', '運輸': '#3b82f6', '小売': '#10b981'}
ordered = summary.sort_values('annual_volatility')
ax.barh(ordered.index, ordered['annual_volatility'], color=[colors[s] for s in ordered['sector']])
ax.set_xlabel('年率ボラティリティ')
ax.set_title('指定30銘柄 — 年率ボラティリティ（過去1年）')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=s) for s, c in colors.items()])
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for sector, group in summary.groupby('sector'):
    ax.scatter(group['annual_volatility'], group['annual_return'], label=sector, s=80, alpha=0.7, color=colors[sector])
    for name, row in group.iterrows():
        ax.annotate(name, (row['annual_volatility'], row['annual_return']), fontsize=8)
ax.axhline(0, color='gray', linewidth=0.5)
ax.set_xlabel('年率ボラティリティ')
ax.set_ylabel('年率リターン')
ax.set_title('リスク・リターン散布図')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 相関ヒートマップ
corr = returns.corr()
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1, square=True, annot=False, ax=ax, cbar_kws={'shrink': 0.7})
ax.set_title('指定30銘柄 — 日次リターン相関行列')
plt.tight_layout()
plt.show()

## 8. ローリング・ボラティリティ

20日（≒1ヶ月）ローリングで時系列の変化を確認。

In [ ]:
WINDOW = 20
rolling_vol = returns.rolling(WINDOW).std() * np.sqrt(TRADING_DAYS)

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
for ax, sector in zip(axes, ['エンタメ', '運輸', '小売']):
    cols = [n for n in rolling_vol.columns if name_to_sector.get(n) == sector]
    rolling_vol[cols].plot(ax=ax, legend=False, alpha=0.7)
    ax.set_title(f'{sector} — 20日ローリング年率ボラ')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.legend(fontsize=7, ncol=5, loc='upper left')
plt.tight_layout()
plt.show()

## 9. データ書き出し（任意）

In [ ]:
OUT = ROOT / 'analysis' / 'output'
OUT.mkdir(exist_ok=True)
summary.to_csv(OUT / 'summary_metrics.csv')
wide.to_csv(OUT / 'close_prices.csv')
returns.to_csv(OUT / 'daily_returns.csv')
print('saved to', OUT)